# Student Performance Prediction
AIML Summer Internship 2026 - MNNIT Allahabad

## Step 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported')

## Step 2 - Load the Dataset

In [ ]:
df = pd.read_csv('../Dataset/student_performance_updated_1000.csv')

print('Shape of dataset:', df.shape)
df.head()

In [ ]:
# check column names and data types
df.info()

In [ ]:
# basic statistics
df.describe()

## Step 3 - Data Preprocessing

In [ ]:
# check missing values
print('Missing values in each column:')
print(df.isnull().sum())

In [ ]:
# drop columns that are not useful for prediction
df.drop(columns=['StudentID', 'Name'], inplace=True)

# drop duplicate columns
df.drop(columns=['Study Hours', 'Attendance (%)'], inplace=True)

print('Columns now:', list(df.columns))

In [ ]:
# fill missing values
# for number columns use median
df['AttendanceRate'].fillna(df['AttendanceRate'].median(), inplace=True)
df['StudyHoursPerWeek'].fillna(df['StudyHoursPerWeek'].median(), inplace=True)
df['PreviousGrade'].fillna(df['PreviousGrade'].median(), inplace=True)
df['ExtracurricularActivities'].fillna(df['ExtracurricularActivities'].median(), inplace=True)
df['FinalGrade'].fillna(df['FinalGrade'].median(), inplace=True)

# for text columns use mode
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['ParentalSupport'].fillna(df['ParentalSupport'].mode()[0], inplace=True)
df['Online Classes Taken'].fillna(df['Online Classes Taken'].mode()[0], inplace=True)

print('Missing values after filling:')
print(df.isnull().sum())

In [ ]:
# remove duplicate rows
df.drop_duplicates(inplace=True)
print('Shape after removing duplicates:', df.shape)

In [ ]:
# fix invalid values found during EDA
# StudyHoursPerWeek had -5 which is not possible
df['StudyHoursPerWeek'] = df['StudyHoursPerWeek'].clip(lower=0)

# AttendanceRate had 200 which is not possible
df['AttendanceRate'] = df['AttendanceRate'].clip(upper=100)

print('Done fixing invalid values')

In [ ]:
# encode categorical columns
le = LabelEncoder()

df['Gender'] = le.fit_transform(df['Gender'])

# parental support has order so we manually map it
df['ParentalSupport'] = df['ParentalSupport'].map({'Low': 0, 'Medium': 1, 'High': 2})

# convert True/False to 1/0
df['Online Classes Taken'] = df['Online Classes Taken'].astype(int)

print('After encoding:')
df.head()

## Step 4 - Exploratory Data Analysis (EDA)

In [ ]:
# distribution of final grade
plt.figure(figsize=(7, 4))
plt.hist(df['FinalGrade'], bins=15, color='steelblue', edgecolor='black')
plt.title('Distribution of Final Grade')
plt.xlabel('Final Grade')
plt.ylabel('Number of Students')
plt.show()

In [ ]:
# histogram for all columns
df.hist(figsize=(12, 8), bins=15)
plt.suptitle('All Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
# scatter plot - study hours vs final grade
plt.figure(figsize=(7, 4))
plt.scatter(df['StudyHoursPerWeek'], df['FinalGrade'], alpha=0.5, color='green')
plt.title('Study Hours vs Final Grade')
plt.xlabel('Study Hours Per Week')
plt.ylabel('Final Grade')
plt.show()

In [ ]:
# scatter plot - attendance vs final grade
plt.figure(figsize=(7, 4))
plt.scatter(df['AttendanceRate'], df['FinalGrade'], alpha=0.5, color='orange')
plt.title('Attendance Rate vs Final Grade')
plt.xlabel('Attendance Rate')
plt.ylabel('Final Grade')
plt.show()

In [ ]:
# boxplots to check outliers
plt.figure(figsize=(7, 4))
sns.boxplot(y=df['StudyHoursPerWeek'])
plt.title('Boxplot - Study Hours Per Week')
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(y=df['AttendanceRate'])
plt.title('Boxplot - Attendance Rate')
plt.show()

In [ ]:
# heatmap to see correlation between features
plt.figure(figsize=(9, 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## Step 5 - Feature Engineering

In [ ]:
# create a new feature combining attendance and study hours
df['AcademicIndex'] = df['AttendanceRate'] + df['StudyHoursPerWeek']

print('New feature AcademicIndex added')
df.head()

## Step 6 - Model Building

In [ ]:
# split data into features and target
X = df.drop(columns=['FinalGrade'])
y = df['FinalGrade']

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training size:', X_train.shape[0])
print('Testing size:', X_test.shape[0])

In [ ]:
# Model 1 - Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print('Linear Regression trained')

In [ ]:
# Model 2 - Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print('Random Forest trained')

In [ ]:
# Model 3 - XGBoost
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

print('XGBoost trained')

## Step 7 - Model Evaluation

In [ ]:
# evaluate linear regression
print('--- Linear Regression ---')
print('MAE :', round(mean_absolute_error(y_test, lr_pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, lr_pred)), 2))
print('R2  :', round(r2_score(y_test, lr_pred), 2))

In [ ]:
# evaluate random forest
print('--- Random Forest ---')
print('MAE :', round(mean_absolute_error(y_test, rf_pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, rf_pred)), 2))
print('R2  :', round(r2_score(y_test, rf_pred), 2))

In [ ]:
# evaluate xgboost
print('--- XGBoost ---')
print('MAE :', round(mean_absolute_error(y_test, xgb_pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, xgb_pred)), 2))
print('R2  :', round(r2_score(y_test, xgb_pred), 2))

In [ ]:
# compare all models in a table
results = {
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE':  [round(mean_absolute_error(y_test, lr_pred), 2),
             round(mean_absolute_error(y_test, rf_pred), 2),
             round(mean_absolute_error(y_test, xgb_pred), 2)],
    'RMSE': [round(np.sqrt(mean_squared_error(y_test, lr_pred)), 2),
             round(np.sqrt(mean_squared_error(y_test, rf_pred)), 2),
             round(np.sqrt(mean_squared_error(y_test, xgb_pred)), 2)],
    'R2':   [round(r2_score(y_test, lr_pred), 2),
             round(r2_score(y_test, rf_pred), 2),
             round(r2_score(y_test, xgb_pred), 2)]
}

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# plot comparison bar chart
plt.figure(figsize=(8, 5))
plt.bar(results_df['Model'], results_df['R2'], color=['blue', 'green', 'red'])
plt.title('R2 Score Comparison')
plt.ylabel('R2 Score')
plt.xticks(rotation=10)
plt.show()

In [ ]:
# actual vs predicted plot for random forest
plt.figure(figsize=(7, 5))
plt.scatter(y_test, rf_pred, alpha=0.5, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Actual vs Predicted - Random Forest')
plt.xlabel('Actual Grade')
plt.ylabel('Predicted Grade')
plt.show()

## Step 8 - Save the Best Model

In [ ]:
# save the best model using pickle
# random forest gave highest R2 so saving that
with open('../Model/model.pkl', 'wb') as f:
    pickle.dump(rf, f)

# also save the feature column list for streamlit app
feature_cols = list(X.columns)
with open('../Model/features.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print('Model saved')
print('Features:', feature_cols)